In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/gas_clean_complete_subsets_1234561h_2026_09_26.csv')

In [6]:
stargrass = df[df['id_lab'] == 'F25-0008']

In [12]:
stargrass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2,delete
1692,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,-644.101024,164.369645,12.53515,NaN,NaN,NaN,NaN,NaN,NaN,no
1693,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,-644.101024,164.369645,12.53515,NaN,"30,31,32",NaN,NaN,NaN,NaN,no
1694,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,-644.101024,164.369645,12.53515,NaN,"33,34,35",NaN,NaN,NaN,NaN,no
1695,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,-644.101024,164.369645,12.53515,NaN,"36,37,38",NaN,NaN,NaN,NaN,no
1696,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,-644.101024,164.369645,12.53515,NaN,"42,43,47",NaN,NaN,NaN,NaN,no


In [13]:
stargrass.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [9]:
#Columns processing
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete']
numeric_columns = [ 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm']

for df in [stargrass]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [18]:

def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in ["id_lab", "batch"]
    ]

    # Group by id_lab and batch
    grouped = df.groupby(["id_lab", "batch"], observed=True)

    # Keep the first value of each categorical column
    cat_df = grouped[cat_cols].first()

    # Calculate the mean of numeric columns
    mean_df = (
        grouped[numeric_columns]
        .mean()
        .round(2)
    )

    # Count the number of replicates in each group
    mean_df["n_replicates_gas"] = grouped.size()

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result


In [19]:
# Calculate batch averages
stargrass_batch_average = mean_with_replicates(stargrass, category_columns, numeric_columns)


In [20]:
stargrass_batch_average

,id_lab,batch,subset,no,requisitioner,id,tax_order,family,genus,species,...,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,part_fact,ch4_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,n_replicates_gas
0,F25-0008,34,2,102,LMF,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,9.91,15.60,17.59,120.15,2.00,21.14,-2.51,112.26,24.29,12
1,F25-0008,35,2,206,LMF,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,8.05,15.07,16.23,105.80,2.91,17.16,-5197.19,57.83,30.68,18
2,F25-0008,36,2,311,LMF,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,8.91,14.63,15.85,119.86,2.46,19.00,122.16,81.62,29.40,18
3,F25-0008,37,2,416,LMF,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,8.57,15.20,16.14,113.17,2.17,18.27,14070.37,74.58,24.50,12
4,F25-0008,38,2,521,LMF,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,9.59,14.67,15.49,131.99,2.64,20.44,807.47,59.74,34.75,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,F25-0008,99,6,1245,JACOBO ARANGO/ ALEJANDRO MONTOYA,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,10.82,16.20,17.11,134.86,3.06,23.07,0.48,55.91,41.29,2
57,F25-0008,100,6,1401,JACOBO ARANGO/ ALEJANDRO MONTOYA,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,11.12,17.55,17.97,132.02,2.96,23.72,0.56,60.76,39.09,2
58,F25-0008,101,6,1407,JACOBO ARANGO/ ALEJANDRO MONTOYA,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,11.68,15.57,16.66,149.41,2.37,24.90,0.84,71.57,35.54,6
59,F25-0008,102,6,1563,JACOBO ARANGO/ ALEJANDRO MONTOYA,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,...,12.10,16.47,17.05,151.32,2.64,25.80,-0.70,64.69,39.94,6


In [21]:
stargrass_batch_average_target_columns = stargrass_batch_average[['id_lab', 'id', 'batch', 'methane_intensity', 'tddm']]

In [22]:
stargrass_batch_average_target_columns

,id_lab,id,batch,methane_intensity,tddm
0,F25-0008,Sample-8,34,112.26,24.29
1,F25-0008,Sample-8,35,57.83,30.68
2,F25-0008,Sample-8,36,81.62,29.40
3,F25-0008,Sample-8,37,74.58,24.50
4,F25-0008,Sample-8,38,59.74,34.75
...,...,...,...,...,...
56,F25-0008,Sample-8,99,55.91,41.29
57,F25-0008,Sample-8,100,60.76,39.09
58,F25-0008,Sample-8,101,71.57,35.54
59,F25-0008,Sample-8,102,64.69,39.94


In [23]:
stargrass_batch_average_target_columns.to_csv('stargrass_batch_average_target_columns.csv', index=False)